# 01 Data Cleaning


Clean raw transactions and prepare analysis-ready customer transaction data.

In [ ]:
import pandas as pd

raw_path = "../data/raw_customer_transactions.csv"
df = pd.read_csv(raw_path)

df.head()

In [ ]:
df.info()
df.isnull().sum()
df.duplicated().sum()

In [ ]:
df = df.drop_duplicates()
df = df.dropna(subset=["CustomerID"])
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df = df.dropna(subset=["InvoiceDate"])
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
df = df[df["OrderStatus"] == "Completed"]
df["Revenue"] = df["Quantity"] * df["UnitPrice"] * (1 - df["DiscountPercent"] / 100)
df["TransactionMonth"] = df["InvoiceDate"].dt.to_period("M").dt.to_timestamp()
df["CohortMonth"] = df.groupby("CustomerID")["TransactionMonth"].transform("min")
df["CohortIndex"] = (
    (df["TransactionMonth"].dt.year - df["CohortMonth"].dt.year) * 12 +
    (df["TransactionMonth"].dt.month - df["CohortMonth"].dt.month) + 1
)
df.to_csv("../data/cleaned_customer_transactions.csv", index=False)
df.head()